# Маркетинговая кампания 

На кого ориентирована кампания, что будем делать, сколько стоит, ожидаемые KPI.

Используем данные из анализа ЦА (источники 1 и 2 оттуда же) + смотрим примерные цены на рекламу в ВКонтакте и Telegram.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['figure.figsize'] = (10, 5)
print('OK')

## На кого ориентирована кампания

По результатам анализа ЦА — основная аудитория:  
- Фермеры и малые производители (40%+ опрошенных)  
- Возраст 30-55 лет  
- Регионы: ЦФО, ЮФО (Краснодарский край, Ростовская обл.), ПФО  
- Площадки присутствия: ВКонтакте, Telegram-каналы для фермеров, профильные форумы


In [ ]:
# Каналы продвижения и оценка бюджета на 3 месяца запуска
# Источник цен: кабинет ВКонтакте Ads, Telega.in (биржа Telegram), оценки по рынку

channels = {
    'Канал': [
        'VK Ads (таргет)',
        'Telegram-каналы (фермеры)',
        'Контекст Яндекс.Директ',
        'Партнёрство с ярмарками',
        'Email-рассылка по базе МСП'
    ],
    'Бюджет на 3 мес (тыс. руб.)': [90, 45, 60, 30, 15],
    'Охват (чел.)': [25000, 8000, 12000, 3000, 5000],
    'Ожидаемых регистраций': [300, 120, 180, 60, 80]
}

df_marketing = pd.DataFrame(channels)
df_marketing['CPL (руб./регистрация)'] = (
    df_marketing['Бюджет на 3 мес (тыс. руб.)'] * 1000 /
    df_marketing['Ожидаемых регистраций']
).round(0).astype(int)

print(df_marketing.to_string(index=False))
print(f"\nОбщий бюджет: {df_marketing['Бюджет на 3 мес (тыс. руб.)'].sum()} тыс. руб.")
print(f"Всего регистраций: {df_marketing['Ожидаемых регистраций'].sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: бюджет по каналам
colors = ['#4472C4', '#ED7D31', '#A9D18E', '#FFC000', '#5B9BD5']
axes[0].pie(
    df_marketing['Бюджет на 3 мес (тыс. руб.)'],
    labels=df_marketing['Канал'],
    autopct='%1.0f%%',
    colors=colors,
    startangle=90
)
axes[0].set_title('Распределение бюджета по каналам')

# График 2: CPL (стоимость одной регистрации)
short_names = ['VK Ads', 'Telegram', 'Яндекс.Директ', 'Ярмарки', 'Email']
bars = axes[1].bar(short_names, df_marketing['CPL (руб./регистрация)'],
                   color=colors, edgecolor='white')
axes[1].set_title('CPL — стоимость одной регистрации (руб.)')
axes[1].set_ylabel('Руб.')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(int(bar.get_height())), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/marketing_budget.png', dpi=120)
plt.show()

best = df_marketing.loc[df_marketing['CPL (руб./регистрация)'].idxmin(), 'Канал']
print(f'Самый дешёвый канал по CPL: {best}')

In [ ]:
# Воронка конверсии
funnel_stages = ['Охват', 'Переходы на сайт', 'Регистрации', 'Активные пользователи', 'Платящие']
funnel_values = [53000, 5300, 740, 370, 74]
conversions = ['-', '10%', '14%', '50%', '20%']

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(funnel_stages))
bar_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(funnel_stages)))

bars = ax.barh(list(y_pos), funnel_values, color=bar_colors, edgecolor='white', height=0.6)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(funnel_stages, fontsize=11)
ax.set_xlabel('Количество человек')
ax.set_title('Маркетинговая воронка EcoLogistic (3 месяца)')

for i, (bar, conv) in enumerate(zip(bars, conversions)):
    ax.text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2,
            f'{funnel_values[i]:,}  ({conv})', va='center', fontsize=10)

ax.set_xlim(0, 62000)
plt.tight_layout()
plt.savefig('../data/funnel_chart.png', dpi=120)
plt.show()

In [ ]:
# KPI и финансовая оценка

paying_users = 74
avg_deliveries_per_month = 25   # из анализа опроса (медиана)
avg_delivery_cost = 2500        # руб., средняя стоимость доставки
commission_rate = 0.07          # 7% — наша комиссия

monthly_revenue = paying_users * avg_deliveries_per_month * avg_delivery_cost * commission_rate
total_budget = df_marketing['Бюджет на 3 мес (тыс. руб.)'].sum() * 1000

kpi_data = {
    'KPI': [
        'Охват за 3 мес.',
        'Регистраций производителей',
        'Активных пользователей',
        'Платящих клиентов',
        'Выручка в месяц (к концу 3-го мес.)',
        'Общий бюджет кампании',
        'CAC (стоимость привлечения платящего)'
    ],
    'Значение': [
        '53 000 чел.',
        '740',
        '370',
        '74',
        f'{monthly_revenue:,.0f} руб.',
        f'{total_budget:,.0f} руб.',
        f'{total_budget/paying_users:,.0f} руб.'
    ]
}

df_kpi = pd.DataFrame(kpi_data)
print('=== KPI маркетинговой кампании ===')
print(df_kpi.to_string(index=False))

---
## Выводы

1. **Приоритетные каналы**: VK Ads (самый большой охват) + Telegram-каналы для фермеров (целевая аудитория).
2. **Бюджет**: 240 тыс. руб. на 3 месяца — реалистично для стартапа на ранней стадии.
3. **Самый эффективный канал по CPL**: Telegram (375 руб./регистрация) — аудитория более целевая, меньше «случайных» кликов.
4. **Воронка**: при бюджете 240 тыс. ожидаем 74 платящих клиентов и ~**323 тыс. руб./мес. выручки** к концу 3-го месяца.
5. **CAC** ≈ 3 240 руб. — при среднем чеке (~4 375 руб./мес. с клиента) окупаемость привлечения меньше месяца. Хороший показатель для B2B.
